In [ ]:
import pandas as pd
df = pd.read_parquet("../data/processed/karachi_events_clean.parquet")

# Surprise 1: the 3 leftover date prefixes
left = df[df["summary_clean"].str.match(r"^\d{2}/\d{2}/\d{4}")]
print(left[["eventid", "summary"]].assign(summary=left["summary"].str[:70]).to_string())

In [ ]:
# Surprise 2: duplicate groups whose events fall on different days
multi = df[df.groupby("dup_group")["eventid"].transform("size") > 1]
days = multi.groupby("dup_group")["date"].nunique()
print("Groups spanning more than one day:", (days > 1).sum())
odd = multi[multi["dup_group"].isin(days[days > 1].index)]
print(odd[["eventid", "date", "summary_clean"]].assign(summary_clean=odd["summary_clean"].str[:70]).to_string())

In [ ]:
pairs = {tuple(sorted((int(e), int(r))))
         for e, rs in zip(df["eventid"], df["related_in_scope"]) for r in rs}
print("Distinct related pairs:", len(pairs))

text = dict(zip(df["eventid"].astype(int), df["summary_clean"]))
trivial = sum(text[a] == text[b] for a, b in pairs)
print("Pairs with identical text (trivial):", trivial)
print("Pairs with different text (a real test):", len(pairs) - trivial)